# Evaluation — Environmental Anomaly Detector

Compare precision/recall across Isolation Forest, LOF, and LSTM Autoencoder.
Ensemble voting analysis for final anomaly classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve

## Evaluation Steps

1. Load anomaly predictions from all three methods
2. Compute precision, recall, and F1 for each method
3. Build ensemble voting classifier (majority vote)
4. Compare PR curves across methods
5. Analyze temporal patterns of detected anomalies

In [ ]:
# Load predictions
results = pd.read_parquet('../data/processed/anomaly_predictions.parquet')
y_true = results['is_anomaly'].values

methods = ['iso_forest', 'lof', 'lstm_ae']
for method in methods:
    y_pred = (results[f'{method}_score'] > results[f'{method}_threshold']).astype(int)
    p = precision_score(y_true, y_pred)
    r = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f'{method:12s}  Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}')

In [ ]:
# Ensemble voting
votes = np.zeros(len(results))
for method in methods:
    votes += (results[f'{method}_score'] > results[f'{method}_threshold']).astype(int)

ensemble_pred = (votes >= 2).astype(int)  # majority vote
p_ens = precision_score(y_true, ensemble_pred)
r_ens = recall_score(y_true, ensemble_pred)
f1_ens = f1_score(y_true, ensemble_pred)
print(f'Ensemble      Precision={p_ens:.3f}  Recall={r_ens:.3f}  F1={f1_ens:.3f}')